# Kalshi & Polymarket Event Study Data Extraction

**Source:** J. Becker  
**Purpose:** Extract market snapshots and trade histories for event study analysis across Kalshi (binary prediction market) and Polymarket (AMM-based market) platforms.

## Overview

This notebook:
1. Downloads compressed archival data (trades, blocks, market snapshots from 2023–2026)
2. Identifies markets active around key event dates (e.g., Ukraine-Russia news, Zelensky lawsuit resolution)
3. Extracts **price snapshots** at 5 am and 5 pm PT on each day in a rolling window
4. Produces both **long-format audit tables** (one row per market-time pair) and **wide-format tables** (one row per market with time-series prices as columns)
5. Generates reference data dictionaries describing column metadata

## Data Caveat

All market snapshots in the archive are **ex-post** (fetched in late February 2026), not collected in real-time during the event windows. Volume/liquidity figures represent full-history values, not contemporaneous measures during the event period. These are a **first-pass approximation** of market activity and interest.

In [ ]:
# =========================================================================
# SECTION: Data Download & Extraction
# =========================================================================
from data_downloader import download_and_extract_data

URL = "https://s3.jbecker.dev/data.tar.zst"
ARCHIVE_NAME = "data.tar.zst"
OUTPUT_DIR = "."

# Call the function to download and extract data
download_and_extract_data(URL, ARCHIVE_NAME, OUTPUT_DIR)

In [ ]:
# =========================================================================
# SECTION: Data Integrity Check & Clean Extraction (Alternative)
# =========================================================================
from data_downloader import verify_and_clean_extraction

URL = "https://s3.jbecker.dev/data.tar.zst"
ARCHIVE_NAME = "data.tar.zst"
OUTPUT_DIR = "."
FORCE_CLEAN = False

# Call the function to verify and clean extraction
verify_and_clean_extraction(URL, ARCHIVE_NAME, OUTPUT_DIR, force_clean=FORCE_CLEAN)

remote_size = 36,020,641,508
local_size  = 36,020,641,508
Done.


In [ ]:
from pathlib import Path
import requests
from tqdm import tqdm

URL = "https://s3.jbecker.dev/data.tar.zst"
archive = Path("data.tar.zst")

if archive.exists():
    print(f"{archive} already exists; skipping download.")
else:
    with requests.get(URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(archive, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    print(f"Downloaded {archive.stat().st_size:,} bytes")
    print(f"Expected   {total:,} bytes")

100%|██████████| 36.0G/36.0G [20:54<00:00, 28.7MB/s]  

Downloaded 36,020,641,508 bytes
Expected   36,020,641,508 bytes


## Data Coverage Exploration

Before extracting event-study data, we run diagnostic queries to check:
- **Date ranges** covered by market snapshots, trades, and blockchain blocks
- **Volume** of data in specific windows (e.g., March–April 2025)
- **Number of active markets** during target event periods

These help confirm that the archive has sufficient coverage for your analysis window.

In [ ]:
# =========================================================================
# SECTION: Market Data Coverage
# =========================================================================
from data_processing import analyze_market_data_for_all

# Define parameters
start_date = "2025-03-01"
end_date = "2025-05-01"

# Analyze data for both Polymarket and Kalshi
analyze_market_data_for_all(start_date, end_date)

market files: 41
trade files:  40454
block files:  785

=== POLYMARKET MARKETS RANGE ===
                  min_created_at                   max_created_at              min_end_date              max_end_date             min_fetched_at             max_fetched_at  n_rows
2020-10-02 09:10:01.467000-07:00 2026-02-03 14:25:08.795666-08:00 2011-07-05 13:21:00-07:00 2028-11-06 16:00:00-08:00 2026-02-03 22:24:36.935776 2026-02-03 22:32:11.641553  408863

=== BLOCK TIMESTAMP RANGE ===
       min_block_ts        max_block_ts   n_rows
2020-09-03 04:33:11 2026-02-02 18:27:46 78468431

=== TRADE TIMESTAMP RANGE ===
       min_trade_ts        max_trade_ts  n_trade_rows
2023-03-05 17:06:59 2026-01-25 17:26:06     404540000

=== TRADES IN MAR-APR 2025 ===
 trades_in_window first_trade_in_window last_trade_in_window
         15500473   2025-03-01 00:00:01  2025-04-30 23:59:59

=== MARKETS TOUCHING MAR-APR 2025 ===
 markets_touching_window
                   53227


## Kalshi Data Coverage Diagnostics

Similar checks for Kalshi markets:
- Market lifecycle (creation → open → close timestamps)
- Trade timestamp range
- Activity during March–April 2025

In [ ]:
# ============================================================================
# SECTION: Kalshi Market & Trade Coverage
# ============================================================================
# Check Kalshi market lifecycle and trade distribution
# ============================================================================

from pathlib import Path
import glob
import duckdb

root = Path("data/kalshi")

market_files = glob.glob(str(root / "markets" / "**" / "*.parquet"), recursive=True)
trade_files  = glob.glob(str(root / "trades" / "**" / "*.parquet"), recursive=True)

print(f"market files: {len(market_files)}")
print(f"trade files:  {len(trade_files)}")

con = duckdb.connect()

# 1) market coverage
if market_files:
    q_markets = """
    SELECT
        MIN(CAST(created_time AS TIMESTAMP))  AS min_created_time,
        MAX(CAST(created_time AS TIMESTAMP))  AS max_created_time,
        MIN(CAST(open_time AS TIMESTAMP))     AS min_open_time,
        MAX(CAST(open_time AS TIMESTAMP))     AS max_open_time,
        MIN(CAST(close_time AS TIMESTAMP))    AS min_close_time,
        MAX(CAST(close_time AS TIMESTAMP))    AS max_close_time,
        MIN(CAST(_fetched_at AS TIMESTAMP))   AS min_fetched_at,
        MAX(CAST(_fetched_at AS TIMESTAMP))   AS max_fetched_at,
        COUNT(*)                              AS n_rows
    FROM read_parquet(?)
    """
    print("\n=== KALSHI MARKETS RANGE ===")
    print(con.execute(q_markets, [market_files]).fetchdf().to_string(index=False))

# 2) trade coverage
if trade_files:
    q_trades = """
    SELECT
        MIN(CAST(created_time AS TIMESTAMP)) AS min_trade_ts,
        MAX(CAST(created_time AS TIMESTAMP)) AS max_trade_ts,
        COUNT(*)                             AS n_trade_rows
    FROM read_parquet(?)
    """
    print("\n=== KALSHI TRADE TIMESTAMP RANGE ===")
    print(con.execute(q_trades, [trade_files]).fetchdf().to_string(index=False))

# 3) specifically check March-April 2025 trades
if trade_files:
    q_trade_window = """
    SELECT
        COUNT(*) AS trades_in_window,
        MIN(CAST(created_time AS TIMESTAMP)) AS first_trade_in_window,
        MAX(CAST(created_time AS TIMESTAMP)) AS last_trade_in_window
    FROM read_parquet(?)
    WHERE CAST(created_time AS TIMESTAMP) >= TIMESTAMP '2025-03-01'
      AND CAST(created_time AS TIMESTAMP) <  TIMESTAMP '2025-05-01'
    """
    print("\n=== KALSHI TRADES IN MAR-APR 2025 ===")
    print(con.execute(q_trade_window, [trade_files]).fetchdf().to_string(index=False))

# 4) markets whose lifecycle touches March-April 2025
if market_files:
    q_market_window = """
    SELECT
        COUNT(*) AS markets_touching_window
    FROM read_parquet(?)
    WHERE
        COALESCE(CAST(close_time AS TIMESTAMP), TIMESTAMP '2100-01-01') >= TIMESTAMP '2025-03-01'
        AND COALESCE(CAST(created_time AS TIMESTAMP), TIMESTAMP '1900-01-01') < TIMESTAMP '2025-05-01'
    """
    print("\n=== KALSHI MARKETS TOUCHING MAR-APR 2025 ===")
    print(con.execute(q_market_window, [market_files]).fetchdf().to_string(index=False))

market files: 769
trade files:  7214

=== KALSHI MARKETS RANGE ===
          min_created_time           max_created_time       min_open_time       max_open_time      min_close_time      max_close_time             min_fetched_at             max_fetched_at  n_rows
2021-06-30 06:46:45.154903 2025-11-23 10:51:48.656951 2021-06-30 07:00:00 2026-11-30 07:00:00 2021-07-01 16:00:00 2099-07-31 21:59:00 2025-11-23 18:51:48.805101 2025-11-24 02:40:25.762553 7682445

=== KALSHI TRADE TIMESTAMP RANGE ===
              min_trade_ts               max_trade_ts  n_trade_rows
2021-06-30 13:09:14.185137 2025-11-25 14:00:15.194245      72134741

=== KALSHI TRADES IN MAR-APR 2025 ===
 trades_in_window      first_trade_in_window       last_trade_in_window
          5052548 2025-03-01 05:00:00.648214 2025-04-30 23:59:58.791842

=== KALSHI MARKETS TOUCHING MAR-APR 2025 ===
 markets_touching_window
                  565940


In [ ]:
# ============================================================================
# SECTION: Kalshi Snapshot Distribution Check (March-April 2025)
# ============================================================================
# Examine how many distinct market snapshots exist per date during the
# March-April 2025 window. This helps understand the granularity and
# frequency of market data collection.
# ============================================================================

import glob
import duckdb

market_files = glob.glob("data/kalshi/markets/**/*.parquet", recursive=True)
con = duckdb.connect()

q = """
WITH x AS (
    SELECT
        ticker,
        CAST(_fetched_at AS TIMESTAMP) AS fetched_ts,
        CAST(_fetched_at AS DATE)      AS fetched_date,
        volume,
        volume_24h,
        open_interest
    FROM read_parquet(?)
    WHERE CAST(_fetched_at AS TIMESTAMP) >= TIMESTAMP '2025-03-01'
      AND CAST(_fetched_at AS TIMESTAMP) <  TIMESTAMP '2025-05-01'
),
per_market AS (
    SELECT
        ticker,
        COUNT(*) AS rows_per_market
    FROM x
    GROUP BY ticker
),
overall AS (
    SELECT
        COUNT(*) AS snapshot_rows,
        COUNT(DISTINCT ticker) AS unique_markets,
        COUNT(DISTINCT fetched_date) AS unique_snapshot_dates
    FROM x
)
SELECT
    overall.snapshot_rows,
    overall.unique_markets,
    overall.unique_snapshot_dates,
    AVG(per_market.rows_per_market) AS avg_rows_per_market
FROM overall
CROSS JOIN per_market
GROUP BY
    overall.snapshot_rows,
    overall.unique_markets,
    overall.unique_snapshot_dates
"""
print(con.execute(q, [market_files]).fetchdf().to_string(index=False))

Empty DataFrame
Columns: [snapshot_rows, unique_markets, unique_snapshot_dates, avg_rows_per_market]
Index: []


In [ ]:
# ============================================================================
# SECTION: Kalshi Snapshot Timeline (March-April 2025)
# ============================================================================
# Count unique markets per date in March-April 2025. Shows when and how
# frequently market data was collected.
# ============================================================================

import glob
import duckdb

market_files = glob.glob("data/kalshi/markets/**/*.parquet", recursive=True)
con = duckdb.connect()

q = """
SELECT
    CAST(_fetched_at AS DATE) AS fetched_date,
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS unique_markets
FROM read_parquet(?)
GROUP BY 1
ORDER BY 1
"""
print(con.execute(q, [market_files]).fetchdf().to_string(index=False))

fetched_date    rows  unique_markets
  2025-11-23 6124800         6124800
  2025-11-24 1557645         1557645


In [ ]:
# ============================================================================
# SECTION: Top-100 Markets by Volume (March 24, 2025 Event Window)
# ============================================================================
# Extract the top 100 most-active markets (by volume) that were trading
# during the March 24, 2025 event window.
#
# Note: Volume values are **ex-post** snapshot values (fetched in Feb 2026),
# not contemporaneous measures from March 2025. This is a first-pass proxy
# for relative market activity/interest. Markets selected by which were
# created ≤ 2 weeks before the event and remained open ≥ 2 weeks after.
# ============================================================================

from pathlib import Path
import glob
import duckdb

# Settings for March 24, 2025 event (Ukraine-Russia news)
EVENT_DATE = "2025-03-24"
CREATED_CUTOFF = "2025-03-10"   # Markets must exist by this date
END_CUTOFF = "2025-04-07"       # Markets must remain open until this date

pm_files = glob.glob("data/polymarket/markets/**/*.parquet", recursive=True)
kalshi_files = glob.glob("data/kalshi/markets/**/*.parquet", recursive=True)

outdir = Path("exports")
outdir.mkdir(exist_ok=True)

con = duckdb.connect()

# ----------------------------
# Polymarket: Top-100 by volume
# ----------------------------
pm_sql = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_at AS TIMESTAMP) AS created_ts,
        TRY_CAST(end_date   AS TIMESTAMP) AS end_ts,
        TRY_CAST(volume     AS DOUBLE)    AS volume_num
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(end_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
)
SELECT *
FROM filtered
ORDER BY volume_num DESC NULLS LAST, id
LIMIT 100
"""

pm_df = con.execute(pm_sql, [pm_files, CREATED_CUTOFF, END_CUTOFF]).fetchdf()

pm_path = outdir / "polymarket_top100_active_around_2025-03-24_snapshot_volume.csv"
pm_df.to_csv(pm_path, index=False)

# ----------------------------
# Kalshi: Top-100 by volume
# ----------------------------
kalshi_sql = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_time AS TIMESTAMP) AS created_ts,
        TRY_CAST(close_time   AS TIMESTAMP) AS close_ts,
        TRY_CAST(volume       AS DOUBLE)    AS volume_num
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(close_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
)
SELECT *
FROM filtered
ORDER BY volume_num DESC NULLS LAST, ticker
LIMIT 100
"""

kalshi_df = con.execute(kalshi_sql, [kalshi_files, CREATED_CUTOFF, END_CUTOFF]).fetchdf()

kalshi_path = outdir / "kalshi_top100_active_around_2025-03-24_snapshot_volume.csv"
kalshi_df.to_csv(kalshi_path, index=False)

print(f"Polymarket rows: {len(pm_df)} -> {pm_path}")
print(f"Kalshi rows:     {len(kalshi_df)} -> {kalshi_path}")

# Optional: quick preview of key columns if present
for name, df, cols in [
    ("Polymarket", pm_df, ["id", "question", "slug", "created_at", "end_date", "volume"]),
    ("Kalshi", kalshi_df, ["ticker", "title", "created_time", "close_time", "volume", "open_interest"]),
]:
    keep = [c for c in cols if c in df.columns]
    print(f"\n{name} preview:")
    print(df[keep].head(10).to_string(index=False))

Polymarket rows: 100 -> exports\polymarket_top100_active_around_2025-03-24_snapshot_volume.csv
Kalshi rows:     100 -> exports\kalshi_top100_active_around_2025-03-24_snapshot_volume.csv

Polymarket preview:
    id                                             question                                                slug                       created_at                  end_date       volume
507892   Will the Sacramento Kings win the 2025 NBA Finals?   will-the-sacramento-kings-win-the-2025-nba-finals 2024-09-24 08:33:30.214289-07:00 2025-06-23 05:00:00-07:00 3.780115e+08
507894    Will the Toronto Raptors win the 2025 NBA Finals?    will-the-toronto-raptors-win-the-2025-nba-finals 2024-09-24 08:34:33.830533-07:00 2025-06-23 05:00:00-07:00 1.542203e+08
507286      Will Aston Villa win the UEFA Champions League?      will-aston-villa-win-the-uefa-champions-league 2024-09-17 10:21:16.527578-07:00 2025-05-31 05:00:00-07:00 1.331138e+08
507896 Will the Washington Wizards win the 2025 NBA Final

In [ ]:
# ============================================================================
# SECTION: Data Dictionary Generation
# ============================================================================
# Build reference metadata for the top-100 market snapshots.
# Each data dictionary CSV contains:
#   - column_name: Name of the field
#   - dtype: Python/pandas data type
#   - non_null_count, null_count: Value coverage
#   - n_unique: Cardinality (for categorical fields)
#   - example_values: Sample values from the column
#   - description: Field definition
# ============================================================================

import pandas as pd

# Manual column descriptions mapping (for reference)
# -------------------------------------------------
pm_desc = {
    "id": "Polymarket market identifier.",
    "condition_id": "Conditional token framework condition identifier.",
    "slug": "Human-readable market slug.",
    "question": "Market question text.",
    "outcomes": "List/string of possible outcomes.",
    "outcome_prices": "Snapshot outcome prices at fetch time.",
    "volume": "Snapshot cumulative volume field from archive market table.",
    "liquidity": "Snapshot liquidity field from archive market table.",
    "active": "Whether market was active at fetch time.",
    "closed": "Whether market was closed at fetch time.",
    "end_date": "Scheduled market end/resolve time.",
    "created_at": "Market creation timestamp.",
    "_fetched_at": "Timestamp when this market row was fetched into the archive.",
    "clob_token_ids": "Associated CLOB token identifiers.",
    "market_maker_address": "Market maker / contract-related address if available.",
    "created_ts": "Helper column: parsed created_at timestamp.",
    "end_ts": "Helper column: parsed end_date timestamp.",
    "volume_num": "Helper column: numeric version of volume used for sorting."
}

kalshi_desc = {
    "ticker": "Kalshi market ticker.",
    "event_ticker": "Kalshi event-level ticker grouping related contracts.",
    "market_type": "Kalshi market type/category.",
    "title": "Market title.",
    "yes_sub_title": "YES contract subtitle/label.",
    "no_sub_title": "NO contract subtitle/label.",
    "status": "Market status at fetch time.",
    "result": "Settled result if available.",
    "yes_bid": "Snapshot YES bid at fetch time.",
    "yes_ask": "Snapshot YES ask at fetch time.",
    "no_bid": "Snapshot NO bid at fetch time.",
    "no_ask": "Snapshot NO ask at fetch time.",
    "last_price": "Snapshot last traded price at fetch time.",
    "volume": "Snapshot cumulative volume field from archive market table.",
    "volume_24h": "Snapshot trailing 24h volume at fetch time.",
    "open_interest": "Snapshot open interest at fetch time.",
    "created_time": "Market creation timestamp.",
    "open_time": "Market open timestamp.",
    "close_time": "Market close/settlement timestamp.",
    "_fetched_at": "Timestamp when this market row was fetched into the archive.",
    "created_ts": "Helper column: parsed created_time timestamp.",
    "close_ts": "Helper column: parsed close_time timestamp.",
    "volume_num": "Helper column: numeric version of volume used for sorting."
}

# Helper function to build reference dictionaries
# -------------------------------------------------------
def make_data_dictionary(df: pd.DataFrame, desc_map: dict, dataset_name: str) -> pd.DataFrame:
    """
    Generate a metadata table describing columns in a DataFrame.
    
    Args:
        df: Input DataFrame to document
        desc_map: Dict mapping column names to human-readable descriptions
        dataset_name: Label for the dataset (used as reference)
    
    Returns:
        DataFrame with columns: dataset, column_name, dtype, non_null_count,
                                null_count, n_unique, example_values, description
    """
    rows = []
    for col in df.columns:
        s = df[col]
        non_null = int(s.notna().sum())
        n_unique = int(s.nunique(dropna=True))
        example_vals = s.dropna().astype(str).head(3).tolist()
        example = " | ".join(example_vals)

        rows.append({
            "dataset": dataset_name,
            "column_name": col,
            "dtype": str(s.dtype),
            "non_null_count": non_null,
            "null_count": int(s.isna().sum()),
            "n_unique": n_unique,
            "example_values": example,
            "description": desc_map.get(col, "No manual description added.")
        })
    return pd.DataFrame(rows)

# -------------------------------------------------
# build + export dictionaries
# -------------------------------------------------
pm_dict = make_data_dictionary(pm_df, pm_desc, "polymarket_top100_active_around_2025-03-24")
kalshi_dict = make_data_dictionary(kalshi_df, kalshi_desc, "kalshi_top100_active_around_2025-03-24")

pm_dict_path = outdir / "polymarket_top100_active_around_2025-03-24_data_dictionary.csv"
kalshi_dict_path = outdir / "kalshi_top100_active_around_2025-03-24_data_dictionary.csv"

pm_dict.to_csv(pm_dict_path, index=False)
kalshi_dict.to_csv(kalshi_dict_path, index=False)

print(f"Polymarket dictionary -> {pm_dict_path}")
print(f"Kalshi dictionary     -> {kalshi_dict_path}")

Polymarket dictionary -> exports\polymarket_top100_active_around_2025-03-24_data_dictionary.csv
Kalshi dictionary     -> exports\kalshi_top100_active_around_2025-03-24_data_dictionary.csv


## Event Studies: Price Snapshots Around Known Dates

For each event study, we:
1. **Define a time window**: 1 week before the event through 1 week after
2. **Build daily cutoffs**: 5 am and 5 pm Pacific Time in the window
3. **Pull trades**: For each market, find the most recent trade price at each cutoff time using `merge_asof` (backward/"as-of" snapshot logic)
4. **Output formats**:
   - **Snapshots (long)**: One row per (market, cutoff_time), shows the trade timestamp and price nearest to that cutoff
   - **Wide**: One row per market, with columns for each cutoff's price (easier for cross-market time-series analysis)
5. **Parallel extraction**: Top-100 markets (by volume proxy) and topic-filtered markets (Ukraine-related keywords)

**Time zone note**: All cutoffs are computed in Pacific Time (`America/Los_Angeles`), then converted to UTC for trade queries.

In [ ]:
# ============================================================================
# SECTION: Price Snapshots at Daily Cutoffs (Top-100 Markets, Mar 24 Event)
# ============================================================================
# For the top-100 active markets, extract price history via "as-of" snapshot
# logic: for each day in a rolling window (1 week before ± 1 week after the
# event), find the most recent trade at or before 5 am PT and 5 pm PT.
#
# Output formats:
#   - Audit (long): One row per (market, time_cutoff) pair, showing the
#     most recent price at that cutoff (if any trade exists).
#   - Wide: One row per market, with columns for each time-cutoff's price,
#     enabling easy time-series comparisons.
# ============================================================================

from pathlib import Path
import ast
import glob
import json
import re
from decimal import Decimal, InvalidOperation, getcontext

import duckdb
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================
EVENT_DATE_LOCAL = pd.Timestamp("2025-03-24", tz="America/Los_Angeles")
START_DATE_LOCAL = (EVENT_DATE_LOCAL - pd.Timedelta(days=7)).normalize()   # 2025-03-17
END_DATE_LOCAL   = (EVENT_DATE_LOCAL + pd.Timedelta(days=7)).normalize()   # 2025-03-31

# input CSVs created earlier
PM_TOP_CSV = Path("exports/polymarket_top100_active_around_2025-03-24_snapshot_volume.csv")
KALSHI_TOP_CSV = Path("exports/kalshi_top100_active_around_2025-03-24_snapshot_volume.csv")

# parquet roots
PM_TRADE_FILES = glob.glob("data/polymarket/trades/**/*.parquet", recursive=True)
PM_BLOCK_FILES = glob.glob("data/polymarket/blocks/**/*.parquet", recursive=True)
KALSHI_TRADE_FILES = glob.glob("data/kalshi/trades/**/*.parquet", recursive=True)

OUTDIR = Path("exports")
OUTDIR.mkdir(exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def parse_list_like(x):
    """Parse JSON-ish / Python-list-ish cell into a Python list."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x

    s = str(x).strip()
    if s == "":
        return []

    try:
        out = json.loads(s)
        if isinstance(out, list):
            return out
    except Exception:
        pass

    try:
        out = ast.literal_eval(s)
        if isinstance(out, list):
            return out
    except Exception:
        pass

    return []


def slugify(s):
    """Safe-ish column suffix for outcome names."""
    if s is None:
        return "outcome"
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "outcome"


def build_cutoffs(start_date_local, end_date_local):
    """
    Build 5am and 5pm America/Los_Angeles cutoffs for each date in [start, end].
    
    Returns:
        DataFrame with columns: cutoff_local, cutoff_utc, cutoff_label
        where cutoff_label is "YYYY_MM_DD_open_0500_PT" or "YYYY_MM_DD_close_1700_PT"
    """
    dates = pd.date_range(start=start_date_local, end=end_date_local, freq="D", tz="America/Los_Angeles")
    rows = []
    for d in dates:
        for hour, tag in [(5, "open_0500_PT"), (17, "close_1700_PT")]:
            cutoff_local = d.normalize() + pd.Timedelta(hours=hour)
            rows.append({
                "cutoff_local": cutoff_local,
                "cutoff_utc": cutoff_local.tz_convert("UTC"),
                "cutoff_label": f"{d.strftime('%Y_%m_%d')}_{tag}",
            })
    return pd.DataFrame(rows)


def asof_snapshots(
    trades_df,
    targets_df,
    by_col,
    trade_time_col="trade_ts_utc",
    target_time_col="cutoff_utc",
    price_col="price",
):
    """
    For each target timestamp, grab the most recent trade at or before the cutoff.
    
    Important: merge_asof requires the time keys to be globally sorted.
    
    Args:
        trades_df: DataFrame with [by_col, trade_time_col, price_col]
        targets_df: DataFrame with [by_col, target_time_col, ...other columns...]
        by_col: Column name to group by (e.g., "ticker" or "token_id")
        
    Returns:
        DataFrame with each target matched to its most recent preceding trade.
    """
    left = (
        targets_df
        .dropna(subset=[by_col, target_time_col])
        .copy()
    )
    right = (
        trades_df
        .dropna(subset=[by_col, trade_time_col])
        .copy()
    )

    left[target_time_col] = pd.to_datetime(left[target_time_col], utc=True)
    right[trade_time_col] = pd.to_datetime(right[trade_time_col], utc=True)

    left = left.sort_values([target_time_col, by_col]).reset_index(drop=True)
    right = right.sort_values([trade_time_col, by_col]).reset_index(drop=True)

    snap = pd.merge_asof(
        left,
        right,
        left_on=target_time_col,
        right_on=trade_time_col,
        by=by_col,
        direction="backward",
        allow_exact_matches=True,
    )
    return snap


def infer_minimum_tick_size(prices: pd.Series) -> float:
    """Infer the minimum tick size (most common decimal increment) from a series of prices."""
    getcontext().prec = 28

    def tick_for_price(x):
        try:
            d = Decimal(str(x)).normalize()
        except (InvalidOperation, ValueError, TypeError):
            return None
        exp = d.as_tuple().exponent
        if exp >= 0:
            return Decimal(1)
        return Decimal(10) ** exp

    ticks = [tick_for_price(x) for x in prices.dropna()]
    ticks = [t for t in ticks if t is not None]
    if not ticks:
        return 1.0
    mode_tick = max(set(ticks), key=ticks.count)
    return float(mode_tick)


def apply_minimum_tick_size_rounding(df: pd.DataFrame, id_col: str = "id", price_col: str = "price") -> pd.DataFrame:
    """Add minimum_tick_size per market and round prices to that tick size."""
    tick_map = (
        df.groupby(id_col)[price_col]
        .apply(infer_minimum_tick_size)
        .reset_index(name="minimum_tick_size")
    )
    df = df.merge(tick_map, on=id_col, how="left")

    mask = df[price_col].notna() & df["minimum_tick_size"].notna() & (df["minimum_tick_size"] > 0)
    df.loc[mask, price_col] = (
        (df.loc[mask, price_col] / df.loc[mask, "minimum_tick_size"])
        .round()
        * df.loc[mask, "minimum_tick_size"]
    )

    return df


# ============================================================
# CUT OFFS
# ============================================================
cutoffs = build_cutoffs(START_DATE_LOCAL, END_DATE_LOCAL)

FINAL_CUTOFF_UTC_STR = (
    cutoffs["cutoff_utc"]
    .max()
    .tz_convert("UTC")
    .tz_localize(None)
    .strftime("%Y-%m-%d %H:%M:%S")
)

print(f"Using local window: {START_DATE_LOCAL.date()} through {END_DATE_LOCAL.date()}")
print(f"Final cutoff UTC:   {FINAL_CUTOFF_UTC_STR}")

con = duckdb.connect()

# ============================================================
# KALSHI
# ============================================================
kalshi_top = pd.read_csv(KALSHI_TOP_CSV)
kalshi_top["ticker"] = kalshi_top["ticker"].astype(str)

kalshi_keys = pd.DataFrame({"ticker": sorted(kalshi_top["ticker"].dropna().astype(str).unique())})
con.register("kalshi_keys", kalshi_keys)

kalshi_sql = """
SELECT
    t.ticker,
    CAST(t.created_time AS TIMESTAMP) AS trade_ts,
    CAST(t.yes_price AS DOUBLE) / 100.0 AS price
FROM read_parquet(?) t
JOIN kalshi_keys k
  ON t.ticker = k.ticker
WHERE CAST(t.created_time AS TIMESTAMP) <= CAST(? AS TIMESTAMP)
"""

kalshi_trades = con.execute(kalshi_sql, [KALSHI_TRADE_FILES, FINAL_CUTOFF_UTC_STR]).fetchdf()
kalshi_trades["ticker"] = kalshi_trades["ticker"].astype(str)
kalshi_trades["trade_ts_utc"] = pd.to_datetime(kalshi_trades["trade_ts"], utc=True)

kalshi_targets = (
    kalshi_keys.assign(_tmp=1)
    .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
    .drop(columns="_tmp")
)
kalshi_targets["cutoff_utc"] = pd.to_datetime(kalshi_targets["cutoff_utc"], utc=True)

kalshi_snaps = asof_snapshots(
    trades_df=kalshi_trades[["ticker", "trade_ts_utc", "price"]],
    targets_df=kalshi_targets[["ticker", "cutoff_utc", "cutoff_label", "cutoff_local"]],
    by_col="ticker",
)

kalshi_audit = kalshi_snaps.copy()
kalshi_audit["source_trade_ts_local"] = kalshi_audit["trade_ts_utc"].dt.tz_convert("America/Los_Angeles")
kalshi_audit_path = OUTDIR / "kalshi_top100_week_before_after_2025_03_24_snapshots_long.csv"
kalshi_audit.to_csv(kalshi_audit_path, index=False)

kalshi_wide = (
    kalshi_snaps.pivot(index="ticker", columns="cutoff_label", values="price")
    .reset_index()
)

kalshi_wide = kalshi_wide.rename(
    columns={c: f"yes_{c}" for c in kalshi_wide.columns if c != "ticker"}
)

kalshi_aug = kalshi_top.merge(kalshi_wide, on="ticker", how="left")
kalshi_aug_path = OUTDIR / "kalshi_top100_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv"
kalshi_aug.to_csv(kalshi_aug_path, index=False)

print(f"Kalshi trades pulled: {len(kalshi_trades):,}")
print(f"Kalshi audit long -> {kalshi_audit_path}")
print(f"Kalshi wide      -> {kalshi_aug_path}")

# ============================================================
# POLYMARKET
# ============================================================
pm_top = pd.read_csv(PM_TOP_CSV)
pm_top["id"] = pm_top["id"].astype(str)

token_rows = []
for _, row in pm_top.iterrows():
    market_id = str(row["id"])
    token_ids = parse_list_like(row.get("clob_token_ids"))
    outcomes = parse_list_like(row.get("outcomes"))

    for i, tok in enumerate(token_ids):
        tok_str = str(tok)
        outcome_name = outcomes[i] if i < len(outcomes) else f"outcome_{i}"
        token_rows.append({
            "id": market_id,
            "token_id": tok_str,
            "outcome_index": i,
            "outcome_name": outcome_name,
            "outcome_slug": slugify(outcome_name),
        })

token_map = pd.DataFrame(token_rows).drop_duplicates()

if token_map.empty:
    raise ValueError("No Polymarket token mapping found. Check that clob_token_ids exists in your CSV.")

con.register("pm_token_map", token_map)

pm_sql = """
WITH trades AS (
    SELECT
        CASE
            WHEN CAST(maker_asset_id AS VARCHAR) = '0'
                THEN CAST(taker_asset_id AS VARCHAR)
            ELSE CAST(maker_asset_id AS VARCHAR)
        END AS token_id,
        block_number,
        CASE
            WHEN CAST(maker_asset_id AS VARCHAR) = '0'
                THEN CAST(maker_amount AS DOUBLE) / NULLIF(CAST(taker_amount AS DOUBLE), 0)
            ELSE CAST(taker_amount AS DOUBLE) / NULLIF(CAST(maker_amount AS DOUBLE), 0)
        END AS price
    FROM read_parquet(?)
),
blocks AS (
    SELECT
        block_number,
        CAST(timestamp AS TIMESTAMP) AS block_ts
    FROM read_parquet(?)
    WHERE CAST(timestamp AS TIMESTAMP) <= CAST(? AS TIMESTAMP)
)
SELECT
    m.id,
    m.token_id,
    m.outcome_index,
    m.outcome_name,
    m.outcome_slug,
    b.block_ts,
    t.price
FROM trades t
JOIN pm_token_map m
  ON t.token_id = m.token_id
JOIN blocks b
  ON t.block_number = b.block_number
"""

pm_trades = con.execute(pm_sql, [PM_TRADE_FILES, PM_BLOCK_FILES, FINAL_CUTOFF_UTC_STR]).fetchdf()

pm_trades["id"] = pm_trades["id"].astype(str)
pm_trades["token_id"] = pm_trades["token_id"].astype(str)
pm_trades["trade_ts_utc"] = pd.to_datetime(pm_trades["block_ts"], utc=True)

pm_targets = (
    token_map.assign(_tmp=1)
    .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
    .drop(columns="_tmp")
)
pm_targets["cutoff_utc"] = pd.to_datetime(pm_targets["cutoff_utc"], utc=True)

pm_snaps = asof_snapshots(
    trades_df=pm_trades[["token_id", "trade_ts_utc", "price"]],
    targets_df=pm_targets[["id", "token_id", "outcome_index", "outcome_name", "outcome_slug", "cutoff_utc", "cutoff_label", "cutoff_local"]],
    by_col="token_id",
)

pm_snaps = apply_minimum_tick_size_rounding(pm_snaps, id_col="id", price_col="price")

pm_audit = pm_snaps.copy()
pm_audit["source_trade_ts_local"] = pm_audit["trade_ts_utc"].dt.tz_convert("America/Los_Angeles")
pm_audit_path = OUTDIR / "polymarket_top100_week_before_after_2025_03_24_snapshots_long.csv"
pm_audit.to_csv(pm_audit_path, index=False)

pm_snaps["wide_col"] = pm_snaps.apply(
    lambda r: f"outcome_{int(r['outcome_index'])}_{r['outcome_slug']}_{r['cutoff_label']}",
    axis=1
)

pm_wide = (
    pm_snaps.pivot(index="id", columns="wide_col", values="price")
    .reset_index()
)
pm_wide = pm_wide.merge(
    pm_snaps[["id", "minimum_tick_size"]].drop_duplicates(subset=["id"]),
    on="id",
    how="left",
)

outcome_name_wide = (
    token_map.assign(name_col=lambda df: df["outcome_index"].map(lambda i: f"outcome_{i}_name"))
    .pivot(index="id", columns="name_col", values="outcome_name")
    .reset_index()
)

pm_aug = pm_top.merge(outcome_name_wide, on="id", how="left").merge(pm_wide, on="id", how="left")
pm_aug_path = OUTDIR / "polymarket_top100_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv"
pm_aug.to_csv(pm_aug_path, index=False)

print(f"Polymarket trades pulled: {len(pm_trades):,}")
print(f"Polymarket audit long -> {pm_audit_path}")
print(f"Polymarket wide      -> {pm_aug_path}")

print("\nDone.")

Using local window: 2025-03-17 through 2025-03-31
Final cutoff UTC:   2025-04-01 00:00:00
Kalshi trades pulled: 216,579
Kalshi audit long -> exports\kalshi_top100_week_before_after_2025_03_24_snapshots_long.csv
Kalshi wide      -> exports\kalshi_top100_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv
Polymarket trades pulled: 14,952,476
Polymarket audit long -> exports\polymarket_top100_week_before_after_2025_03_24_snapshots_long.csv
Polymarket wide      -> exports\polymarket_top100_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv

Done.


## Topic-Filtered Markets: Ukraine-Russia Events

Rather than using full-period top-100 rankings, this section filters for markets explicitly related to Ukraine, Russia, and regional geopolitics. Using a regex search across market titles/descriptions, we extract:
- Markets that match topic keywords (ukraine, russia, putin, nato, etc.)
- Trade history and price snapshots during the same event window (5 am/5 pm PT)

**Note:** Different markets may have different trading patterns and activity levels during the event period compared to the aggregated top-100 set.

In [ ]:
# ============================================================================
# SECTION: Topic-Filtered Markets & Snapshots (Ukraine-Russia Events)
# ============================================================================
# Filter for markets related to Ukraine, Russia, and geopolitical events
# using regex on market titles/descriptions. Uses the same snapshot logic
# (5 am / 5 pm PT) but applied to these topic-filtered markets instead of
# top-100 aggregates.
# ============================================================================

from pathlib import Path
import ast
import glob
import json
import re

import duckdb
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================
EVENT_DATE_LOCAL = pd.Timestamp("2025-03-24", tz="America/Los_Angeles")
START_DATE_LOCAL = (EVENT_DATE_LOCAL - pd.Timedelta(days=7)).normalize()   # 2025-03-17
END_DATE_LOCAL   = (EVENT_DATE_LOCAL + pd.Timedelta(days=7)).normalize()   # 2025-03-31

CREATED_CUTOFF = "2025-03-10"
END_CUTOFF = "2025-04-07"

# Topic patterns: case-insensitive whole-word regex terms
# Each term is a word boundary regex like \bukraine\b matching "ukraine" anywhere
TOPIC_PATTERNS = [
    r"\bukraine\b",
    r"\bukrainian\b",
    r"\brussia\b",
    r"\brussian\b",
    r"\bputin\b",
    r"\bzelensky\b",
    r"\bzelenskyy\b",
    r"\bzelenskyyy\b",
    r"\bzelenskyj\b",
    r"\bkyiv\b",
    r"\bkiev\b",
    r"\bmoscow\b",
    r"\bkremlin\b",
    r"\bdonbas\b",
    r"\bdonetsk\b",
    r"\bluhansk\b",
    r"\bcrimea\b",
    r"\bsevastopol\b",
    r"\bkharkiv\b",
    r"\bodesa\b",
    r"\bodessa\b",
    r"\bzaporizhzhia\b",
    r"\bzaporizhia\b",
    r"\bkherson\b",
    r"\bmariupol\b",
    r"\bnato\b",
]

TOPIC_REGEX = "|".join(TOPIC_PATTERNS)

PM_MARKET_FILES = glob.glob("data/polymarket/markets/**/*.parquet", recursive=True)
PM_TRADE_FILES = glob.glob("data/polymarket/trades/**/*.parquet", recursive=True)
PM_BLOCK_FILES = glob.glob("data/polymarket/blocks/**/*.parquet", recursive=True)

KALSHI_MARKET_FILES = glob.glob("data/kalshi/markets/**/*.parquet", recursive=True)
KALSHI_TRADE_FILES = glob.glob("data/kalshi/trades/**/*.parquet", recursive=True)

OUTDIR = Path("exports")
OUTDIR.mkdir(exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def parse_list_like(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "":
        return []
    try:
        out = json.loads(s)
        if isinstance(out, list):
            return out
    except Exception:
        pass
    try:
        out = ast.literal_eval(s)
        if isinstance(out, list):
            return out
    except Exception:
        pass
    return []


def slugify(s):
    if s is None:
        return "outcome"
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "outcome"


def build_cutoffs(start_date_local, end_date_local):
    dates = pd.date_range(start=start_date_local, end=end_date_local, freq="D", tz="America/Los_Angeles")
    rows = []
    for d in dates:
        for hour, tag in [(5, "open_0500_PT"), (17, "close_1700_PT")]:
            cutoff_local = d.normalize() + pd.Timedelta(hours=hour)
            rows.append({
                "cutoff_local": cutoff_local,
                "cutoff_utc": cutoff_local.tz_convert("UTC"),
                "cutoff_label": f"{d.strftime('%Y_%m_%d')}_{tag}",
            })
    return pd.DataFrame(rows)


def asof_snapshots(
    trades_df,
    targets_df,
    by_col,
    trade_time_col="trade_ts_utc",
    target_time_col="cutoff_utc",
):
    left = targets_df.dropna(subset=[by_col, target_time_col]).copy()
    right = trades_df.dropna(subset=[by_col, trade_time_col]).copy()

    left[target_time_col] = pd.to_datetime(left[target_time_col], utc=True)
    right[trade_time_col] = pd.to_datetime(right[trade_time_col], utc=True)

    left = left.sort_values([target_time_col, by_col]).reset_index(drop=True)
    right = right.sort_values([trade_time_col, by_col]).reset_index(drop=True)

    snap = pd.merge_asof(
        left,
        right,
        left_on=target_time_col,
        right_on=trade_time_col,
        by=by_col,
        direction="backward",
        allow_exact_matches=True,
    )
    return snap


cutoffs = build_cutoffs(START_DATE_LOCAL, END_DATE_LOCAL)
FINAL_CUTOFF_UTC_STR = (
    cutoffs["cutoff_utc"]
    .max()
    .tz_convert("UTC")
    .tz_localize(None)
    .strftime("%Y-%m-%d %H:%M:%S")
)

print(f"Using local window: {START_DATE_LOCAL.date()} through {END_DATE_LOCAL.date()}")
print(f"Final cutoff UTC:   {FINAL_CUTOFF_UTC_STR}")

con = duckdb.connect()

# ============================================================
# 1) FILTER MARKETS BY TOPIC + LIFECYCLE WINDOW
# ============================================================

# ---------- Polymarket ----------
pm_sql = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_at AS TIMESTAMP) AS created_ts,
        TRY_CAST(end_date   AS TIMESTAMP) AS end_ts,
        LOWER(
            COALESCE(CAST(question AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(slug AS VARCHAR), '')
        ) AS search_text
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(end_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
      AND regexp_matches(search_text, ?)
)
SELECT *
FROM filtered
ORDER BY created_ts, id
"""

pm_markets = con.execute(
    pm_sql,
    [PM_MARKET_FILES, CREATED_CUTOFF, END_CUTOFF, TOPIC_REGEX],
).fetchdf()

pm_base_path = OUTDIR / "polymarket_ukraine_russia_active_around_2025-03-24.csv"
pm_markets.to_csv(pm_base_path, index=False)

print(f"Polymarket topic markets: {len(pm_markets):,} -> {pm_base_path}")

# ---------- Kalshi ----------
kalshi_sql = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_time AS TIMESTAMP) AS created_ts,
        TRY_CAST(close_time   AS TIMESTAMP) AS close_ts,
        LOWER(
            COALESCE(CAST(title AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(ticker AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(event_ticker AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(yes_sub_title AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(no_sub_title AS VARCHAR), '')
        ) AS search_text
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(close_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
      AND regexp_matches(search_text, ?)
)
SELECT *
FROM filtered
ORDER BY created_ts, ticker
"""

kalshi_markets = con.execute(
    kalshi_sql,
    [KALSHI_MARKET_FILES, CREATED_CUTOFF, END_CUTOFF, TOPIC_REGEX],
).fetchdf()

kalshi_base_path = OUTDIR / "kalshi_ukraine_russia_active_around_2025-03-24.csv"
kalshi_markets.to_csv(kalshi_base_path, index=False)

print(f"Kalshi topic markets:     {len(kalshi_markets):,} -> {kalshi_base_path}")

# ============================================================
# 2) KALSHI SNAPSHOTS
# ============================================================
if len(kalshi_markets) > 0:
    kalshi_markets["ticker"] = kalshi_markets["ticker"].astype(str)
    kalshi_keys = pd.DataFrame({"ticker": sorted(kalshi_markets["ticker"].dropna().astype(str).unique())})
    con.register("kalshi_keys", kalshi_keys)

    kalshi_trade_sql = """
    SELECT
        t.ticker,
        CAST(t.created_time AS TIMESTAMP) AS trade_ts,
        CAST(t.yes_price AS DOUBLE) / 100.0 AS price
    FROM read_parquet(?) t
    JOIN kalshi_keys k
      ON t.ticker = k.ticker
    WHERE CAST(t.created_time AS TIMESTAMP) <= CAST(? AS TIMESTAMP)
    """

    kalshi_trades = con.execute(
        kalshi_trade_sql,
        [KALSHI_TRADE_FILES, FINAL_CUTOFF_UTC_STR],
    ).fetchdf()

    kalshi_trades["ticker"] = kalshi_trades["ticker"].astype(str)
    kalshi_trades["trade_ts_utc"] = pd.to_datetime(kalshi_trades["trade_ts"], utc=True)

    kalshi_targets = (
        kalshi_keys.assign(_tmp=1)
        .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
        .drop(columns="_tmp")
    )
    kalshi_targets["cutoff_utc"] = pd.to_datetime(kalshi_targets["cutoff_utc"], utc=True)

    kalshi_snaps = asof_snapshots(
        trades_df=kalshi_trades[["ticker", "trade_ts_utc", "price"]],
        targets_df=kalshi_targets[["ticker", "cutoff_utc", "cutoff_label", "cutoff_local"]],
        by_col="ticker",
    )

    kalshi_audit = kalshi_snaps.copy()
    kalshi_audit["source_trade_ts_local"] = kalshi_audit["trade_ts_utc"].dt.tz_convert("America/Los_Angeles")
    kalshi_audit_path = OUTDIR / "kalshi_ukraine_russia_week_before_after_2025_03_24_snapshots_long.csv"
    kalshi_audit.to_csv(kalshi_audit_path, index=False)

    kalshi_wide = (
        kalshi_snaps.pivot(index="ticker", columns="cutoff_label", values="price")
        .reset_index()
    )
    kalshi_wide = kalshi_wide.rename(
        columns={c: f"yes_{c}" for c in kalshi_wide.columns if c != "ticker"}
    )

    kalshi_aug = kalshi_markets.merge(kalshi_wide, on="ticker", how="left")
    kalshi_aug_path = OUTDIR / "kalshi_ukraine_russia_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv"
    kalshi_aug.to_csv(kalshi_aug_path, index=False)

    print(f"Kalshi trades pulled: {len(kalshi_trades):,}")
    print(f"Kalshi audit long -> {kalshi_audit_path}")
    print(f"Kalshi wide      -> {kalshi_aug_path}")
else:
    print("No Kalshi topic markets matched the filter.")

# ============================================================
# 3) POLYMARKET SNAPSHOTS
# ============================================================
if len(pm_markets) > 0:
    pm_markets["id"] = pm_markets["id"].astype(str)

    token_rows = []
    for _, row in pm_markets.iterrows():
        market_id = str(row["id"])
        token_ids = parse_list_like(row.get("clob_token_ids"))
        outcomes = parse_list_like(row.get("outcomes"))

        for i, tok in enumerate(token_ids):
            tok_str = str(tok)
            outcome_name = outcomes[i] if i < len(outcomes) else f"outcome_{i}"
            token_rows.append({
                "id": market_id,
                "token_id": tok_str,
                "outcome_index": i,
                "outcome_name": outcome_name,
                "outcome_slug": slugify(outcome_name),
            })

    token_map = pd.DataFrame(token_rows).drop_duplicates()

    if token_map.empty:
        raise ValueError("No Polymarket token mapping found. Check clob_token_ids / outcomes in the filtered file.")

    con.register("pm_token_map", token_map)

    pm_trade_sql = """
    WITH trades AS (
        SELECT
            CASE
                WHEN CAST(maker_asset_id AS VARCHAR) = '0'
                    THEN CAST(taker_asset_id AS VARCHAR)
                ELSE CAST(maker_asset_id AS VARCHAR)
            END AS token_id,
            block_number,
            CASE
                WHEN CAST(maker_asset_id AS VARCHAR) = '0'
                    THEN CAST(maker_amount AS DOUBLE) / NULLIF(CAST(taker_amount AS DOUBLE), 0)
                ELSE CAST(taker_amount AS DOUBLE) / NULLIF(CAST(maker_amount AS DOUBLE), 0)
            END AS price
        FROM read_parquet(?)
    ),
    blocks AS (
        SELECT
            block_number,
            CAST(timestamp AS TIMESTAMP) AS block_ts
        FROM read_parquet(?)
        WHERE CAST(timestamp AS TIMESTAMP) <= CAST(? AS TIMESTAMP)
    )
    SELECT
        m.id,
        m.token_id,
        m.outcome_index,
        m.outcome_name,
        m.outcome_slug,
        b.block_ts,
        t.price
    FROM trades t
    JOIN pm_token_map m
      ON t.token_id = m.token_id
    JOIN blocks b
      ON t.block_number = b.block_number
    """

    pm_trades = con.execute(
        pm_trade_sql,
        [PM_TRADE_FILES, PM_BLOCK_FILES, FINAL_CUTOFF_UTC_STR],
    ).fetchdf()

    pm_trades["id"] = pm_trades["id"].astype(str)
    pm_trades["token_id"] = pm_trades["token_id"].astype(str)
    pm_trades["trade_ts_utc"] = pd.to_datetime(pm_trades["block_ts"], utc=True)

    pm_targets = (
        token_map.assign(_tmp=1)
        .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
        .drop(columns="_tmp")
    )
    pm_targets["cutoff_utc"] = pd.to_datetime(pm_targets["cutoff_utc"], utc=True)

    if "apply_minimum_tick_size_rounding" not in globals():
        from decimal import Decimal, InvalidOperation, getcontext

        def infer_minimum_tick_size(prices: pd.Series) -> float:
            """Infer the minimum tick size (most common decimal increment) from prices."""
            getcontext().prec = 28

            def tick_for_price(x):
                try:
                    d = Decimal(str(x)).normalize()
                except (InvalidOperation, ValueError, TypeError):
                    return None
                exp = d.as_tuple().exponent
                if exp >= 0:
                    return Decimal(1)
                return Decimal(10) ** exp

            ticks = [tick_for_price(x) for x in prices.dropna()]
            ticks = [t for t in ticks if t is not None]
            if not ticks:
                return 1.0
            mode_tick = max(set(ticks), key=ticks.count)
            return float(mode_tick)


        def apply_minimum_tick_size_rounding(df: pd.DataFrame, id_col: str = "id", price_col: str = "price") -> pd.DataFrame:
            """Add minimum_tick_size per market and round prices to that tick size."""
            tick_map = (
                df.groupby(id_col)[price_col]
                .apply(infer_minimum_tick_size)
                .reset_index(name="minimum_tick_size")
            )
            df = df.merge(tick_map, on=id_col, how="left")

            mask = df[price_col].notna() & df["minimum_tick_size"].notna() & (df["minimum_tick_size"] > 0)
            df.loc[mask, price_col] = (
                (df.loc[mask, price_col] / df.loc[mask, "minimum_tick_size"])
                .round()
                * df.loc[mask, "minimum_tick_size"]
            )

            return df

    pm_snaps = asof_snapshots(
        trades_df=pm_trades[["token_id", "trade_ts_utc", "price"]],
        targets_df=pm_targets[["id", "token_id", "outcome_index", "outcome_name", "outcome_slug", "cutoff_utc", "cutoff_label", "cutoff_local"]],
        by_col="token_id",
    )

    pm_snaps = apply_minimum_tick_size_rounding(pm_snaps, id_col="id", price_col="price")

    pm_audit = pm_snaps.copy()
    pm_audit["source_trade_ts_local"] = pm_audit["trade_ts_utc"].dt.tz_convert("America/Los_Angeles")
    pm_audit_path = OUTDIR / "polymarket_ukraine_russia_week_before_after_2025_03_24_snapshots_long.csv"
    pm_audit.to_csv(pm_audit_path, index=False)

    pm_snaps["wide_col"] = pm_snaps.apply(
        lambda r: f"outcome_{int(r['outcome_index'])}_{r['outcome_slug']}_{r['cutoff_label']}",
        axis=1
    )

    pm_wide = (
        pm_snaps.pivot(index="id", columns="wide_col", values="price")
        .reset_index()
    )
    pm_wide = pm_wide.merge(
        pm_snaps[["id", "minimum_tick_size"]].drop_duplicates(subset=["id"]),
        on="id",
        how="left",
    )

    outcome_name_wide = (
        token_map.assign(name_col=lambda df: df["outcome_index"].map(lambda i: f"outcome_{i}_name"))
        .pivot(index="id", columns="name_col", values="outcome_name")
        .reset_index()
    )

    pm_aug = pm_markets.merge(outcome_name_wide, on="id", how="left").merge(pm_wide, on="id", how="left")
    pm_aug_path = OUTDIR / "polymarket_ukraine_russia_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv"
    pm_aug.to_csv(pm_aug_path, index=False)

    print(f"Polymarket trades pulled: {len(pm_trades):,}")
    print(f"Polymarket audit long -> {pm_audit_path}")
    print(f"Polymarket wide      -> {pm_aug_path}")
else:
    print("No Polymarket topic markets matched the filter.")

print("\nDone.")

Using local window: 2025-03-17 through 2025-03-31
Final cutoff UTC:   2025-04-01 00:00:00
Polymarket topic markets: 58 -> exports\polymarket_ukraine_russia_active_around_2025-03-24.csv
Kalshi topic markets:     11 -> exports\kalshi_ukraine_russia_active_around_2025-03-24.csv
Kalshi trades pulled: 7,276
Kalshi audit long -> exports\kalshi_ukraine_russia_week_before_after_2025_03_24_snapshots_long.csv
Kalshi wide      -> exports\kalshi_ukraine_russia_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv
Polymarket trades pulled: 875,920
Polymarket audit long -> exports\polymarket_ukraine_russia_week_before_after_2025_03_24_snapshots_long.csv
Polymarket wide      -> exports\polymarket_ukraine_russia_active_around_2025-03-24_with_open_close_5am_5pm_PT.csv

Done.


## Zelensky-Suit Event Study

This section repeats the earlier extraction workflow for a different event: the Zelensky lawsuit market resolution window. Since that market took multiple days to resolve, we specify both a **resolution start date** and **resolution end date**, then extract price history from one week before the start through one week after the end.

The workflow mirrors the March 24 event study:
- Top-100 markets by volume active during the window
- Topic-filtered (Ukraine-related) markets
- Price snapshots at 5 am and 5 pm PT each day
- Both long (audit) and wide (time-series) CSV outputs

In [ ]:
# ============================================================================
# SECTION: Zelensky-Suit – Settings & Top-100 Extraction
# ============================================================================
# Settings for Zelensky suit market resolution window:
#   - Specify RESOLUTION_START_LOCAL and RESOLUTION_END_LOCAL
#   - Window is automatically 1 week before start through 1 week after end
#   - Then extract top-100 and topic-filtered markets (same logic as March 24)
# ============================================================================

import pandas as pd

# specify the resolution period (update as necessary for different events)
RESOLUTION_START_LOCAL = pd.Timestamp("2025-06-20", tz="America/Los_Angeles")
RESOLUTION_END_LOCAL   = pd.Timestamp("2025-07-08", tz="America/Los_Angeles")

# full window goes one week before start through one week after end
START_DATE_LOCAL = (RESOLUTION_START_LOCAL - pd.Timedelta(days=7)).normalize()
END_DATE_LOCAL   = (RESOLUTION_END_LOCAL   + pd.Timedelta(days=7)).normalize()

CREATED_CUTOFF = START_DATE_LOCAL.strftime("%Y-%m-%d")
END_CUTOFF     = END_DATE_LOCAL.strftime("%Y-%m-%d")

print(f"Zelensky suit resolution: {RESOLUTION_START_LOCAL.date()} to {RESOLUTION_END_LOCAL.date()}")
print(f"Using extraction window: {START_DATE_LOCAL.date()} through {END_DATE_LOCAL.date()}")

# -- top‑100 by liquidity (re‑using earlier SQL templates) --
pm_df = con.execute(pm_sql, [pm_files, CREATED_CUTOFF, END_CUTOFF]).fetchdf()
pm_path = outdir / f"polymarket_top100_zelensky_suit_{START_DATE_LOCAL.date()}_to_{END_DATE_LOCAL.date()}_snapshot_volume.csv"
pm_df.to_csv(pm_path, index=False)

kalshi_df = con.execute(kalshi_sql, [kalshi_files, CREATED_CUTOFF, END_CUTOFF]).fetchdf()
kalshi_path = outdir / f"kalshi_top100_zelensky_suit_{START_DATE_LOCAL.date()}_to_{END_DATE_LOCAL.date()}_snapshot_volume.csv"
kalshi_df.to_csv(kalshi_path, index=False)

print(f"Polymarket rows: {len(pm_df)} -> {pm_path}")
print(f"Kalshi rows:     {len(kalshi_df)} -> {kalshi_path}")

Zelensky suit resolution: 2025-03-20 to 2025-03-27
Using extraction window: 2025-03-13 through 2025-04-03
Polymarket rows: 100 -> exports\polymarket_top100_zelensky_suit_2025-03-13_to_2025-04-03_snapshot_volume.csv
Kalshi rows:     100 -> exports\kalshi_top100_zelensky_suit_2025-03-13_to_2025-04-03_snapshot_volume.csv


In [ ]:
# ============================================================================
# SECTION: Zelensky-Suit – Topic-Filtered Markets & Snapshots
# ============================================================================
# Extract topic-filtered (Ukraine-related) markets for the Zelensky-suit
# resolution window, then pull price snapshots using the same 5 am/5 pm PT
# cutoff logic. Reuse helper functions and the same TOPIC_REGEX from earlier.
# ============================================================================

# helper functions and TOPIC_REGEX are already defined from earlier sections

pm_sql_topic = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_at AS TIMESTAMP) AS created_ts,
        TRY_CAST(end_date   AS TIMESTAMP) AS end_ts,
        LOWER(
            COALESCE(CAST(question AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(slug AS VARCHAR), '')
        ) AS search_text
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(end_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
      AND regexp_matches(search_text, ?)
)
SELECT *
FROM filtered
ORDER BY created_ts, id
"""

pm_markets = con.execute(
    pm_sql_topic,
    [pm_files, CREATED_CUTOFF, END_CUTOFF, TOPIC_REGEX],
).fetchdf()
pm_base_path = outdir / f"polymarket_zelensky_suit_active_{START_DATE_LOCAL.date()}_to_{END_DATE_LOCAL.date()}.csv"
pm_markets.to_csv(pm_base_path, index=False)
print(f"Polymarket topic markets: {len(pm_markets):,} -> {pm_base_path}")

kalshi_sql_topic = """
WITH base AS (
    SELECT
        *,
        TRY_CAST(created_time AS TIMESTAMP) AS created_ts,
        TRY_CAST(close_time   AS TIMESTAMP) AS close_ts,
        LOWER(
            COALESCE(CAST(title AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(ticker AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(event_ticker AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(yes_sub_title AS VARCHAR), '') || ' ' ||
            COALESCE(CAST(no_sub_title AS VARCHAR), '')
        ) AS search_text
    FROM read_parquet(?)
),
filtered AS (
    SELECT *
    FROM base
    WHERE created_ts <= CAST(? AS TIMESTAMP)
      AND COALESCE(close_ts, CAST('2100-01-01' AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
      AND regexp_matches(search_text, ?)
)
SELECT *
FROM filtered
ORDER BY created_ts, ticker
"""

kalshi_markets = con.execute(
    kalshi_sql_topic,
    [KALSHI_MARKET_FILES, CREATED_CUTOFF, END_CUTOFF, TOPIC_REGEX],
).fetchdf()
kalshi_base_path = outdir / f"kalshi_zelensky_suit_active_{START_DATE_LOCAL.date()}_to_{END_DATE_LOCAL.date()}.csv"
kalshi_markets.to_csv(kalshi_base_path, index=False)
print(f"Kalshi topic markets:     {len(kalshi_markets):,} -> {kalshi_base_path}")

# snapshots logic largely identical to the previous topic block; reuse if markets exist
if len(kalshi_markets) > 0:
    kalshi_markets["ticker"] = kalshi_markets["ticker"].astype(str)
    kalshi_keys = pd.DataFrame({"ticker": sorted(kalshi_markets["ticker"].dropna().astype(str).unique())})
    con.register("kalshi_keys", kalshi_keys)

    kalshi_trades = con.execute(
        kalshi_trade_sql,
        [KALSHI_TRADE_FILES, FINAL_CUTOFF_UTC_STR],
    ).fetchdf()
    kalshi_trades["ticker"] = kalshi_trades["ticker"].astype(str)
    kalshi_trades["trade_ts_utc"] = pd.to_datetime(kalshi_trades["trade_ts"], utc=True)

    kalshi_targets = (
        kalshi_keys.assign(_tmp=1)
        .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
        .drop(columns="_tmp")
    )
    kalshi_targets["cutoff_utc"] = pd.to_datetime(kalshi_targets["cutoff_utc"], utc=True)

    kalshi_snaps = asof_snapshots(
        trades_df=kalshi_trades[["ticker", "trade_ts_utc", "price"]],
        targets_df=kalshi_targets[["ticker", "cutoff_utc", "cutoff_label", "cutoff_local"]],
        by_col="ticker",
    )

    kalshi_audit_path = outdir / f"kalshi_zelensky_suit_week_before_after_snapshots_long.csv"
    kalshi_snaps.to_csv(kalshi_audit_path, index=False)

    kalshi_wide = (
        kalshi_snaps.pivot(index="ticker", columns="cutoff_label", values="price")
        .reset_index()
    )
    kalshi_wide = kalshi_wide.rename(columns={c: f"yes_{c}" for c in kalshi_wide.columns if c != "ticker"})
    kalshi_aug = kalshi_markets.merge(kalshi_wide, on="ticker", how="left")
    kalshi_aug_path = outdir / f"kalshi_zelensky_suit_with_open_close_5am_5pm_PT.csv"
    kalshi_aug.to_csv(kalshi_aug_path, index=False)

    print(f"Kalshi trades pulled: {len(kalshi_trades):,}")
    print(f"Kalshi audit long -> {kalshi_audit_path}")
    print(f"Kalshi wide      -> {kalshi_aug_path}")
else:
    print("No Kalshi topic markets matched the filter for Zelensky suit.")

if len(pm_markets) > 0:
    pm_markets["id"] = pm_markets["id"].astype(str)

    token_rows = []
    for _, row in pm_markets.iterrows():
        market_id = str(row["id"])
        token_ids = parse_list_like(row.get("clob_token_ids"))
        outcomes = parse_list_like(row.get("outcomes"))

        for i, tok in enumerate(token_ids):
            tok_str = str(tok)
            outcome_name = outcomes[i] if i < len(outcomes) else f"outcome_{i}"
            token_rows.append({
                "id": market_id,
                "token_id": tok_str,
                "outcome_index": i,
                "outcome_name": outcome_name,
                "outcome_slug": slugify(outcome_name),
            })

    token_map = pd.DataFrame(token_rows).drop_duplicates()
    if token_map.empty:
        raise ValueError("No Polymarket token mapping found. Check clob_token_ids / outcomes in filtered file.")
    con.register("pm_token_map", token_map)

    pm_trades = con.execute(
        pm_trade_sql,
        [PM_TRADE_FILES, PM_BLOCK_FILES, FINAL_CUTOFF_UTC_STR],
    ).fetchdf()
    pm_trades["id"] = pm_trades["id"].astype(str)
    pm_trades["token_id"] = pm_trades["token_id"].astype(str)
    pm_trades["trade_ts_utc"] = pd.to_datetime(pm_trades["block_ts"], utc=True)

    pm_targets = (
        token_map.assign(_tmp=1)
        .merge(cutoffs.assign(_tmp=1), on="_tmp", how="inner")
        .drop(columns="_tmp")
    )
    pm_targets["cutoff_utc"] = pd.to_datetime(pm_targets["cutoff_utc"], utc=True)

    if "apply_minimum_tick_size_rounding" not in globals():
        from decimal import Decimal, InvalidOperation, getcontext

        def infer_minimum_tick_size(prices: pd.Series) -> float:
            """Infer the minimum tick size (most common decimal increment) from prices."""
            getcontext().prec = 28

            def tick_for_price(x):
                try:
                    d = Decimal(str(x)).normalize()
                except (InvalidOperation, ValueError, TypeError):
                    return None
                exp = d.as_tuple().exponent
                if exp >= 0:
                    return Decimal(1)
                return Decimal(10) ** exp

            ticks = [tick_for_price(x) for x in prices.dropna()]
            ticks = [t for t in ticks if t is not None]
            if not ticks:
                return 1.0
            mode_tick = max(set(ticks), key=ticks.count)
            return float(mode_tick)


        def apply_minimum_tick_size_rounding(df: pd.DataFrame, id_col: str = "id", price_col: str = "price") -> pd.DataFrame:
            """Add minimum_tick_size per market and round prices to that tick size."""
            tick_map = (
                df.groupby(id_col)[price_col]
                .apply(infer_minimum_tick_size)
                .reset_index(name="minimum_tick_size")
            )
            df = df.merge(tick_map, on=id_col, how="left")

            mask = df[price_col].notna() & df["minimum_tick_size"].notna() & (df["minimum_tick_size"] > 0)
            df.loc[mask, price_col] = (
                (df.loc[mask, price_col] / df.loc[mask, "minimum_tick_size"])
                .round()
                * df.loc[mask, "minimum_tick_size"]
            )

            return df

    pm_snaps = asof_snapshots(
        trades_df=pm_trades[["token_id", "trade_ts_utc", "price"]],
        targets_df=pm_targets[["id", "token_id", "outcome_index", "outcome_name", "outcome_slug", "cutoff_utc", "cutoff_label", "cutoff_local"]],
        by_col="token_id",
    )

    pm_snaps = apply_minimum_tick_size_rounding(pm_snaps, id_col="id", price_col="price")

    pm_audit_path = outdir / f"polymarket_zelensky_suit_week_before_after_snapshots_long.csv"
    pm_snaps.to_csv(pm_audit_path, index=False)

    pm_snaps["wide_col"] = pm_snaps.apply(
        lambda r: f"outcome_{int(r['outcome_index'])}_{r['outcome_slug']}_{r['cutoff_label']}",
        axis=1,
    )
    pm_wide = (
        pm_snaps.pivot(index="id", columns="wide_col", values="price")
        .reset_index()
    )
    pm_wide = pm_wide.merge(
        pm_snaps[["id", "minimum_tick_size"]].drop_duplicates(subset=["id"]),
        on="id",
        how="left",
    )

    outcome_name_wide = (
        token_map.assign(name_col=lambda df: df["outcome_index"].map(lambda i: f"outcome_{i}_name"))
        .pivot(index="id", columns="name_col", values="outcome_name")
        .reset_index()
    )

    pm_aug = pm_markets.merge(outcome_name_wide, on="id", how="left").merge(pm_wide, on="id", how="left")
    pm_aug_path = outdir / f"polymarket_zelensky_suit_with_open_close_5am_5pm_PT.csv"
    pm_aug.to_csv(pm_aug_path, index=False)

    print(f"Polymarket trades pulled: {len(pm_trades):,}")
    print(f"Polymarket audit long -> {pm_audit_path}")
    print(f"Polymarket wide      -> {pm_aug_path}")
else:
    print("No Polymarket topic markets matched the filter for Zelensky suit.")

Polymarket topic markets: 58 -> exports\polymarket_zelensky_suit_active_2025-03-17_to_2025-03-31.csv
Kalshi topic markets:     11 -> exports\kalshi_zelensky_suit_active_2025-03-17_to_2025-03-31.csv
Kalshi trades pulled: 7,276
Kalshi audit long -> exports\kalshi_zelensky_suit_week_before_after_snapshots_long.csv
Kalshi wide      -> exports\kalshi_zelensky_suit_with_open_close_5am_5pm_PT.csv
Polymarket trades pulled: 875,920
Polymarket audit long -> exports\polymarket_zelensky_suit_week_before_after_snapshots_long.csv
Polymarket wide      -> exports\polymarket_zelensky_suit_with_open_close_5am_5pm_PT.csv
